# Конформная калибровка предсказательных интервалов

Поверх точечной регрессии нутриентов строим интервальные предсказания с
гарантированным маржинальным покрытием на test. Это центральная часть
магистерской: точное число — это точка, тогда как полезная оценка калорий
выглядит как «350 ± 60 ккал, 90 % уверенности». Сравниваем два метода —
split conformal на абсолютных остатках лучшей точечной модели из ноутбука 03
(`gated`) и CQR с обученной квантильной головой на тех же эмбеддингах. Оба
дают теоретическую гарантию покрытия не ниже `1 − α` при допущении
обмениваемости calibration и test, разница — в адаптивности ширины
интервала к сложности конкретного блюда.

## 1. Подготовка окружения

Подключены три input-датасета: `nutrition5k-overhead-rgb-224` (метаданные
и сплиты), `nutrition5k-visual-baseline` (визуальные эмбеддинги и
нормализация целей), `nutrition5k-multimodal-fusion` (точечные предсказания
gated и текстовые эмбеддинги). GPU нужен для обучения квантильной головы;
она маленькая, T4 с запасом.

In [ ]:
import json
import math
import os
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
import seaborn as sns

INPUT_DATA = Path("/kaggle/input/datasets/dbkarashev/nutrition5k-overhead-rgb-224/n5k_overhead")
INPUT_BASELINE = Path("/kaggle/input/datasets/dbkarashev/nutrition5k-visual-baseline/visual_baseline")
INPUT_FUSION = Path("/kaggle/input/datasets/dbkarashev/nutrition5k-multimodal-fusion/multimodal_fusion")

WORK_DIR = Path("/kaggle/working/conformal_calibration")
INT_DIR = WORK_DIR / "intervals"
MODELS_DIR = WORK_DIR / "models"
EDA_DIR = WORK_DIR / "eda"
for d in (WORK_DIR, INT_DIR, MODELS_DIR, EDA_DIR, INT_DIR / "split_cp", INT_DIR / "cqr"):
    d.mkdir(parents=True, exist_ok=True)

CONFORMAL_FILE = WORK_DIR / "conformal_quantiles.json"
METRICS_FILE = WORK_DIR / "metrics.json"
STATUS_FILE = WORK_DIR / "_status.json"

TARGETS = ["total_calories", "total_fat", "total_carb", "total_protein"]
ALPHA = 0.10
TARGET_COVERAGE = 1.0 - ALPHA
QUANTILES = [0.05, 0.50, 0.95]
LOW_IDX, MED_IDX, HIGH_IDX = 0, 1, 2

VIS_DIM = 384
TEXT_DIM = 384
HIDDEN = 256
DROPOUT = 0.2
HEAD_BATCH = 256
HEAD_EPOCHS = 80
HEAD_LR = 1e-3
HEAD_WD = 1e-4
EARLY_STOP = 12
RANDOM_SEED = 42


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")


def update_status(patch: dict) -> None:
    state = {}
    if STATUS_FILE.exists():
        try:
            state = json.loads(STATUS_FILE.read_text())
        except json.JSONDecodeError:
            pass
    state.update(patch)
    state["last_updated"] = now_iso()
    STATUS_FILE.write_text(json.dumps(state, indent=2, ensure_ascii=False))


set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
print(f"alpha={ALPHA}  target coverage={TARGET_COVERAGE:.0%}  quantiles={QUANTILES}")

## 2. Загрузка артефактов из предыдущих ноутбуков

Точечные предсказания берем из `nutrition5k-multimodal-fusion/predictions/gated/*.parquet`.
Эмбеддинги (визуальные и текстовые) — из baseline и fusion соответственно.
Параметры нормализации целей — из baseline. Фактические y_true присутствуют
в gated.parquet, но дополнительно сверяем с dishes.parquet.

In [ ]:
gated_cal = pd.read_parquet(INPUT_FUSION / "predictions" / "gated" / "calibration.parquet")
gated_test = pd.read_parquet(INPUT_FUSION / "predictions" / "gated" / "test.parquet")

vis_cal = np.load(INPUT_BASELINE / "embeddings" / "calibration.npz", allow_pickle=False)
vis_test = np.load(INPUT_BASELINE / "embeddings" / "test.npz", allow_pickle=False)
vis_train = np.load(INPUT_BASELINE / "embeddings" / "train.npz", allow_pickle=False)

text_cal = np.load(INPUT_FUSION / "text_embeddings" / "calibration.npz", allow_pickle=False)
text_test = np.load(INPUT_FUSION / "text_embeddings" / "test.npz", allow_pickle=False)
text_train = np.load(INPUT_FUSION / "text_embeddings" / "train.npz", allow_pickle=False)

norm = json.loads((INPUT_BASELINE / "target_norm.json").read_text())
mean = np.array(norm["mean"], dtype=np.float32)
std = np.array(norm["std"], dtype=np.float32)


def aligned(emb_npz, ids: list[str], emb_name: str) -> np.ndarray:
    cached_ids = list(emb_npz["dish_ids"])
    if cached_ids == ids:
        return emb_npz["embeddings"]
    pos = {d: i for i, d in enumerate(cached_ids)}
    return emb_npz["embeddings"][[pos[d] for d in ids]]


cal_ids = gated_cal["dish_id"].tolist()
test_ids = gated_test["dish_id"].tolist()

V_cal = aligned(vis_cal, cal_ids, "vis_cal")
V_test = aligned(vis_test, test_ids, "vis_test")
T_cal = aligned(text_cal, cal_ids, "text_cal")
T_test = aligned(text_test, test_ids, "text_test")

y_cal = gated_cal[[f"true_{t}" for t in TARGETS]].to_numpy(dtype=np.float32)
y_test = gated_test[[f"true_{t}" for t in TARGETS]].to_numpy(dtype=np.float32)
yhat_cal = gated_cal[[f"pred_{t}" for t in TARGETS]].to_numpy(dtype=np.float32)
yhat_test = gated_test[[f"pred_{t}" for t in TARGETS]].to_numpy(dtype=np.float32)

print(f"  cal:  {len(cal_ids)} блюд,  V={V_cal.shape}, T={T_cal.shape}")
print(f"  test: {len(test_ids)} блюд, V={V_test.shape}, T={T_test.shape}")

## 3. Метод 1: Split conformal на абсолютных остатках

Считаем абсолютные остатки `s = |y - yhat|` на calibration отдельно для
каждой цели. Конформный квантиль уровня `(n+1)(1−α)/n` — это
fine-sample-корректная версия эмпирического квантиля. Финальный интервал
симметричный: `[yhat − q, yhat + q]`. Гарантия покрытия не ниже `1−α`
справедлива при обмене местами строк calibration и test.

In [ ]:
def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    level = math.ceil((n + 1) * (1 - alpha)) / n
    level = min(level, 1.0)
    return float(np.quantile(scores, level, method="higher"))


split_cp_q = {}
for i, t in enumerate(TARGETS):
    s = np.abs(y_cal[:, i] - yhat_cal[:, i])
    split_cp_q[t] = conformal_quantile(s, ALPHA)

split_cp_lo = yhat_test - np.array([split_cp_q[t] for t in TARGETS])
split_cp_hi = yhat_test + np.array([split_cp_q[t] for t in TARGETS])

for t, q in split_cp_q.items():
    print(f"  {t:<18} q = {q:.3f}  (полуширина интервала)")

## 4. Обучение квантильной головы для CQR

Архитектура — та же gated, что в ноутбуке 03, но выход развернут в три
квантиля на каждую цель (`(B, 4, 3)`). Лосс — pinball, усредненный по
квантилям и целям. Чтобы избежать пересечения квантилей (q_lo > q_hi),
выход сортируется по последней оси — это сохраняет градиенты и не требует
дополнительных регуляризаторов. Обучение на нормализованных целях,
валидация и ранняя остановка по pinball на calibration.

In [ ]:
def mlp(in_dim: int, hidden: int, out_dim: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(hidden, out_dim),
    )


class GatedQuantile(nn.Module):
    def __init__(self, n_targets: int, n_quantiles: int):
        super().__init__()
        self.n_t = n_targets
        self.n_q = n_quantiles
        self.v_proj = nn.Linear(VIS_DIM, HIDDEN)
        self.t_proj = nn.Linear(TEXT_DIM, HIDDEN)
        self.gate = nn.Sequential(nn.Linear(2 * HIDDEN, HIDDEN), nn.Sigmoid())
        self.head = mlp(HIDDEN, HIDDEN, n_targets * n_quantiles, DROPOUT)

    def forward(self, v, t):
        v_h = self.v_proj(v)
        t_h = self.t_proj(t)
        g = self.gate(torch.cat([v_h, t_h], dim=-1))
        fused = g * v_h + (1.0 - g) * t_h
        out = self.head(fused).view(-1, self.n_t, self.n_q)
        return torch.sort(out, dim=-1).values


def pinball_multi(pred: torch.Tensor, target: torch.Tensor, quantiles: torch.Tensor) -> torch.Tensor:
    diff = target.unsqueeze(-1) - pred
    return torch.maximum(quantiles * diff, (quantiles - 1.0) * diff).mean()


def make_loader_qr(V: np.ndarray, T: np.ndarray, y_raw: np.ndarray, shuffle: bool) -> DataLoader:
    y_norm = (y_raw - mean) / std
    ds = TensorDataset(torch.from_numpy(V), torch.from_numpy(T), torch.from_numpy(y_norm))
    return DataLoader(ds, batch_size=HEAD_BATCH, shuffle=shuffle, drop_last=False)


CQR_CKPT = MODELS_DIR / "cqr_head.pt"
CQR_HISTORY = MODELS_DIR / "cqr_history.json"


def train_cqr() -> tuple[GatedQuantile, dict]:
    model = GatedQuantile(len(TARGETS), len(QUANTILES)).to(device)
    if CQR_CKPT.exists() and CQR_HISTORY.exists():
        print("[STEP-4] CQR уже обучена, загружаю чекпоинт")
        model.load_state_dict(torch.load(CQR_CKPT, map_location=device))
        return model, json.loads(CQR_HISTORY.read_text())

    print("[STEP-4] Запуск обучения CQR")
    set_seed(RANDOM_SEED)
    model = GatedQuantile(len(TARGETS), len(QUANTILES)).to(device)

    y_train_true = pd.read_parquet(INPUT_DATA / "metadata" / "dishes.parquet").set_index("dish_id") \
        .loc[list(vis_train["dish_ids"])][TARGETS].to_numpy(dtype=np.float32)

    train_loader = make_loader_qr(vis_train["embeddings"], aligned(text_train, list(vis_train["dish_ids"]), "text_train"),
                                  y_train_true, shuffle=True)
    val_loader = make_loader_qr(V_cal, T_cal, y_cal, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LR, weight_decay=HEAD_WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HEAD_EPOCHS)
    q_tensor = torch.tensor(QUANTILES, dtype=torch.float32, device=device).view(1, 1, -1)

    history = {"train_loss": [], "val_loss": []}
    best_val = math.inf
    bad_epochs = 0
    best_state = None

    for epoch in range(1, HEAD_EPOCHS + 1):
        model.train()
        train_losses = []
        for vb, tb, yb in train_loader:
            vb = vb.to(device); tb = tb.to(device); yb = yb.to(device)
            pred = model(vb, tb)
            loss = pinball_multi(pred, yb, q_tensor)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()

        model.eval()
        with torch.inference_mode():
            val_losses = []
            for vb, tb, yb in val_loader:
                vb = vb.to(device); tb = tb.to(device); yb = yb.to(device)
                val_losses.append(pinball_multi(model(vb, tb), yb, q_tensor).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            bad_epochs = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
        if bad_epochs >= EARLY_STOP:
            print(f"  ранняя остановка на эпохе {epoch}, best val pinball {best_val:.4f}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, CQR_CKPT)
    CQR_HISTORY.write_text(json.dumps(history, indent=2))
    return model, history


cqr_model, cqr_history = train_cqr()
update_status({"step_4_cqr_trained": True})

## 5. Метод 2: CQR

На calibration считаем nonconformity score
`E_i = max(qhat_lo(x_i) − y_i, y_i − qhat_hi(x_i))`. Конформный квантиль `Q`
уровня `(n+1)(1−α)/n` от этих скоров — добавочная поправка. Итоговый
интервал: `[qhat_lo(x) − Q, qhat_hi(x) + Q]`. Если квантильная голова уже
калибровочно согласована с данными, поправка `Q` будет около нуля; если
систематически переуверена — `Q` подтянет интервал до нужного покрытия.

In [ ]:
@torch.inference_mode()
def predict_quantiles(V: np.ndarray, T: np.ndarray) -> np.ndarray:
    cqr_model.eval()
    pred = cqr_model(torch.from_numpy(V).to(device), torch.from_numpy(T).to(device)).cpu().numpy()
    return pred * std[None, :, None] + mean[None, :, None]


cqr_cal = predict_quantiles(V_cal, T_cal)
cqr_test = predict_quantiles(V_test, T_test)

cqr_q = {}
for i, t in enumerate(TARGETS):
    qlo = cqr_cal[:, i, LOW_IDX]
    qhi = cqr_cal[:, i, HIGH_IDX]
    scores = np.maximum(qlo - y_cal[:, i], y_cal[:, i] - qhi)
    cqr_q[t] = conformal_quantile(scores, ALPHA)

cqr_lo = cqr_test[:, :, LOW_IDX] - np.array([cqr_q[t] for t in TARGETS])
cqr_hi = cqr_test[:, :, HIGH_IDX] + np.array([cqr_q[t] for t in TARGETS])

CONFORMAL_FILE.write_text(json.dumps({
    "alpha": ALPHA,
    "split_cp_q": split_cp_q,
    "cqr_q": cqr_q,
}, indent=2, ensure_ascii=False))

for t, q in cqr_q.items():
    print(f"  {t:<18} CQR Q = {q:+.3f}")

## 6. Сравнение методов

Главные метрики — маржинальное покрытие (доля test-точек, попавших в
интервал) и средняя ширина. По теории coverage не ниже `1−α`; на практике
с конечными выборками возможны отклонения порядка `1/√n`. Дополнительно
смотрим conditional coverage по квартилям истинного значения — это
показывает, не «дает ли модель гарантию только за счет тяжелых случаев».

In [ ]:
def coverage_and_width(lo: np.ndarray, hi: np.ndarray, y_true: np.ndarray) -> dict:
    out = {}
    for i, t in enumerate(TARGETS):
        cov = float(np.mean((y_true[:, i] >= lo[:, i]) & (y_true[:, i] <= hi[:, i])))
        width = float(np.mean(hi[:, i] - lo[:, i]))
        out[t] = {"coverage": cov, "avg_width": width}
    return out


def conditional_coverage(lo: np.ndarray, hi: np.ndarray, y_true: np.ndarray) -> dict:
    out = {}
    for i, t in enumerate(TARGETS):
        bins = pd.qcut(y_true[:, i], q=4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
        per_bin = {}
        for b in bins.categories:
            mask = np.asarray(bins == b)
            if mask.sum() == 0:
                continue
            cov = float(np.mean((y_true[mask, i] >= lo[mask, i]) & (y_true[mask, i] <= hi[mask, i])))
            per_bin[str(b)] = cov
        out[t] = per_bin
    return out


metrics = {
    "alpha": ALPHA,
    "target_coverage": TARGET_COVERAGE,
    "split_cp": {
        "test": coverage_and_width(split_cp_lo, split_cp_hi, y_test),
        "calibration": coverage_and_width(
            yhat_cal - np.array([split_cp_q[t] for t in TARGETS]),
            yhat_cal + np.array([split_cp_q[t] for t in TARGETS]),
            y_cal,
        ),
        "conditional_test": conditional_coverage(split_cp_lo, split_cp_hi, y_test),
    },
    "cqr": {
        "test": coverage_and_width(cqr_lo, cqr_hi, y_test),
        "conditional_test": conditional_coverage(cqr_lo, cqr_hi, y_test),
    },
}
METRICS_FILE.write_text(json.dumps(metrics, indent=2, ensure_ascii=False))

print(f"Целевое покрытие: {TARGET_COVERAGE:.0%}\n")
print(f"  {'target':<18}{'split CP cov':>14}{'split CP w':>14}{'CQR cov':>14}{'CQR w':>14}")
for t in TARGETS:
    s = metrics["split_cp"]["test"][t]
    c = metrics["cqr"]["test"][t]
    print(f"  {t:<18}{s['coverage']:>14.3f}{s['avg_width']:>14.2f}{c['coverage']:>14.3f}{c['avg_width']:>14.2f}")

update_status({"step_6_evaluated": True})

## 7. Сохранение интервалов и графики

Финальные интервалы по обоим методам сохраняются в parquet и пригодны для
инференса. Графики: покрытие и ширина по целям, conditional coverage по
квартилям, и пример интервалов на калориях (отсортированных по истинному
значению, с error bars).

In [ ]:
def save_intervals(name: str, ids: list[str], lo: np.ndarray, hi: np.ndarray, y_true: np.ndarray) -> None:
    out_dir = INT_DIR / name
    df = pd.DataFrame({"dish_id": ids})
    for i, t in enumerate(TARGETS):
        df[f"lo_{t}"] = lo[:, i]
        df[f"hi_{t}"] = hi[:, i]
        df[f"true_{t}"] = y_true[:, i]
    df.to_parquet(out_dir / "test.parquet", index=False)


save_intervals("split_cp", test_ids, split_cp_lo, split_cp_hi, y_test)
save_intervals("cqr", test_ids, cqr_lo, cqr_hi, y_test)

sns.set_theme(style="whitegrid", context="talk")

x = np.arange(len(TARGETS))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, key, ylabel, title in [
    (axes[0], "coverage", "Coverage", "Маржинальное покрытие на test"),
    (axes[1], "avg_width", "Средняя ширина", "Средняя ширина интервала на test"),
]:
    s_vals = [metrics["split_cp"]["test"][t][key] for t in TARGETS]
    c_vals = [metrics["cqr"]["test"][t][key] for t in TARGETS]
    ax.bar(x - width / 2, s_vals, width, label="split CP", color="steelblue")
    ax.bar(x + width / 2, c_vals, width, label="CQR", color="seagreen")
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace("total_", "") for t in TARGETS])
    ax.set(title=title, ylabel=ylabel)
    ax.legend()
    if key == "coverage":
        ax.axhline(TARGET_COVERAGE, color="red", ls="--", lw=1, label=f"target {TARGET_COVERAGE:.0%}")
        ax.set_ylim(0.7, 1.0)
fig.tight_layout()
fig.savefig(EDA_DIR / "coverage_and_width.png", dpi=120)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, method_name, color in [(axes[0], "split_cp", "steelblue"), (axes[1], "cqr", "seagreen")]:
    cond = metrics[method_name]["conditional_test"]
    bins_order = ["Q1", "Q2", "Q3", "Q4"]
    bw = 0.18
    for j, t in enumerate(TARGETS):
        vals = [cond[t].get(b, np.nan) for b in bins_order]
        ax.bar(np.arange(len(bins_order)) + (j - 1.5) * bw, vals, bw,
               label=t.replace("total_", ""))
    ax.axhline(TARGET_COVERAGE, color="red", ls="--", lw=1)
    ax.set(title=f"{method_name}: покрытие по квартилям истины",
           xticks=np.arange(len(bins_order)), ylim=(0.7, 1.0))
    ax.set_xticklabels(bins_order)
    ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig(EDA_DIR / "conditional_coverage.png", dpi=120)
plt.close(fig)

order = np.argsort(y_test[:, 0])
fig, ax = plt.subplots(figsize=(13, 6))
ax.fill_between(np.arange(len(order)),
                cqr_lo[order, 0], cqr_hi[order, 0],
                alpha=0.3, color="seagreen", label="CQR 90 % интервал")
ax.fill_between(np.arange(len(order)),
                split_cp_lo[order, 0], split_cp_hi[order, 0],
                alpha=0.2, color="steelblue", label="split CP 90 % интервал")
ax.plot(np.arange(len(order)), y_test[order, 0], color="red", lw=1, label="истина")
ax.set(xlabel="блюда test (отсортированы по истинным калориям)",
       ylabel="ккал", title="Калории: сравнение интервалов")
ax.legend()
fig.tight_layout()
fig.savefig(EDA_DIR / "intervals_calories.png", dpi=120)
plt.close(fig)

print(f"  графики в {EDA_DIR}: {sorted(p.name for p in EDA_DIR.glob('*.png'))}")
update_status({"step_7_eda_completed": True})

## 8. Финальный лог

Сводно: какой метод дает покрытие около 90 % при минимальной средней
ширине интервала. После Save Version опубликовать output как
`nutrition5k-conformal-intervals` — это финальный артефакт магистерской
части по неопределенности.

In [ ]:
def avg_width_test(method: str) -> float:
    return float(np.mean([metrics[method]["test"][t]["avg_width"] for t in TARGETS]))


def avg_coverage_test(method: str) -> float:
    return float(np.mean([metrics[method]["test"][t]["coverage"] for t in TARGETS]))


summary = {
    "alpha": ALPHA,
    "target_coverage": TARGET_COVERAGE,
    "n_calibration": len(cal_ids),
    "n_test": len(test_ids),
    "split_cp": {
        "avg_coverage_test": round(avg_coverage_test("split_cp"), 4),
        "avg_width_test": round(avg_width_test("split_cp"), 3),
        "per_target": metrics["split_cp"]["test"],
    },
    "cqr": {
        "avg_coverage_test": round(avg_coverage_test("cqr"), 4),
        "avg_width_test": round(avg_width_test("cqr"), 3),
        "per_target": metrics["cqr"]["test"],
    },
    "verdict": "cqr" if avg_width_test("cqr") < avg_width_test("split_cp") else "split_cp",
    "timestamp": now_iso(),
}
update_status({"step_8_summary_written": True, "summary": summary})

print(json.dumps(summary, indent=2, ensure_ascii=False))
print()
print(f"Лучший метод по средней ширине при покрытии ~{TARGET_COVERAGE:.0%}: {summary['verdict']}")
print("Save & Run All (Commit), затем опубликуй output как nutrition5k-conformal-intervals.")